# JeDepth Training on Kaggle RTX Pro 6000

Setup môi trường, copy code, và huấn luyện mô hình JeDepth stereo depth estimation.

**Workflow:**
1. Import utility packages (pre-installed, no internet needed)
2. Copy code từ dataset → working dir
3. Setup data paths (symlink)
4. Chạy training với tensorboard, eval + test inference mỗi 5 epoch

In [ ]:
import sys, os

# Thêm utility packages vào path (cài sẵn từ jedepth-utility-script kernel)
UTILITY_PATH = "/kaggle/input/jedepth-utility-script"
if os.path.exists(UTILITY_PATH):
    sys.path.insert(0, UTILITY_PATH)

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
# Copy JeDepth code từ dataset (read-only) sang working dir
import shutil

CODE_SRC = "/kaggle/input/jedepth-code"
WORK_DIR = "/kaggle/working/JeDepth"

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
shutil.copytree(CODE_SRC, WORK_DIR)

os.chdir(WORK_DIR)
print(f"Working dir: {os.getcwd()}")
!ls

In [ ]:
# Setup data paths: symlink dataset vào data/processed_data
DATASET_SRC = "/kaggle/input/stereo-smallbaseline"

os.makedirs("data", exist_ok=True)
if not os.path.exists("data/processed_data"):
    os.symlink(DATASET_SRC, "data/processed_data")
print(f"Symlinked: data/processed_data -> {DATASET_SRC}")

# Verify dataset
import pandas as pd
train_df = pd.read_csv("data/processed_data/train.csv")
val_df = pd.read_csv("data/processed_data/val.csv")
print(f"Train samples: {len(train_df)}")
print(f"Val samples:   {len(val_df)}")
print(f"Example path:  {train_df.iloc[0]['left_image_path']}")

!ls data/processed_data/ | head -10

In [ ]:
# Verify imports trước khi training
sys.path.insert(0, WORK_DIR)

from jedepth.model.iinet import JeDepth
from jedepth.dataset.depth_dataset import StereoDepthDataset
from jedepth.evaluation.evaluation import evaluate as compute_metrics
from easydict import EasyDict

print("All JeDepth imports OK!")

In [ ]:
# Chạy training!
# RTX Pro 6000 có ~48GB VRAM → batch_size=8 hoặc lớn hơn
# Thay đổi config nếu cần: sửa cfgs/iinet/iinet_custom.yaml trước khi chạy

# Tùy chỉnh batch size cho GPU lớn
!sed -i 's/BATCH_SIZE: 2/BATCH_SIZE: 8/' cfgs/iinet/iinet_custom.yaml
!sed -i 's/AMP: true/AMP: false/' cfgs/iinet/iinet_custom.yaml
!cat cfgs/iinet/iinet_custom.yaml

print("\n\n=== Starting Training ===")
!python train.py \
    --cfg cfgs/iinet/iinet_custom.yaml \
    --output_dir /kaggle/working/output \
    --workers 8 \
    --gpu 0 \
    --test_images test_images \
    --seed 42

In [ ]:
# Copy outputs sang /kaggle/working để download
import shutil

OUTPUT_SRC = "/kaggle/working/output"
LOGS_DST = "/kaggle/working/results"

if os.path.exists(OUTPUT_SRC):
    shutil.copytree(OUTPUT_SRC, LOGS_DST, dirs_exist_ok=True)
    print(f"Results copied to {LOGS_DST}")

    # List checkpoints
    import glob
    ckpts = sorted(glob.glob(f"{LOGS_DST}/**/ckpt/*.pth", recursive=True))
    print(f"\nCheckpoints ({len(ckpts)}):")
    for c in ckpts:
        size_mb = os.path.getsize(c) / 1024**2
        print(f"  {os.path.basename(c)} ({size_mb:.1f} MB)")
else:
    print("No output found - training may not have completed.")